# 1.4 Tune Regularization Hyperparameters

## Model Cycle: The 5 Key Steps

### 1. Build the Model : Create the pipeline with regularization.  
### 2. Train the Model : Fit the model on the training data.  
### 3. Generate Predictions : Use the trained model to make predictions.  
### 4. Evaluate the Model : Assess performance using evaluation metrics.  
### **5. Improve the Model : Tune hyperparameters for optimal performance.**

## Introduction

In the previous notebooks, we built and trained regularized logistic regression models using default hyperparameter values (`C=1.0`, `l1_ratio=0.5`). However, these default values may not be optimal for our specific dataset.

In this notebook, we use **GridSearchCV** to systematically search for the best hyperparameter values. This is a critical step in the machine learning workflow that can significantly improve model performance.

### Learning Objectives

By the end of this notebook, you will be able to:

1. Use GridSearchCV to tune the regularization strength (`C` parameter)
2. Tune the ElasticNet mixing parameter (`l1_ratio`)
3. Visualize hyperparameter performance
4. Select the best model based on cross-validation results
5. Evaluate the final model on the held-out test set

## 1. Load Dependencies and Data

In [1]:
# Data is in the project's data/ directory
# No Google Drive mount needed when running locally

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import pickle
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix, make_scorer
)

pd.options.display.max_columns = None

In [3]:
# Set up file paths
# Set up file paths
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
models_filepath = '/models/'

In [4]:
# Load training and test data
df_training = pd.read_csv(f'{project_path}{data_filepath}training.csv')
df_testing = pd.read_csv(f'{project_path}{data_filepath}testing.csv')

X_train = df_training
y_train = df_training['SEM_3_STATUS']

X_test = df_testing
y_test = df_testing['SEM_3_STATUS']

print(f"Training data: {X_train.shape[0]} samples")
print(f"Test data: {X_test.shape[0]} samples")
print(f"\nTarget distribution (Training):")
print(y_train.value_counts(normalize=True))

Training data: 19844 samples
Test data: 5336 samples

Target distribution (Training):
SEM_3_STATUS
E    0.867113
N    0.132887
Name: proportion, dtype: float64


In [5]:
# Load the regularized models from Course 3
l2_model = pickle.load(open(f'{project_path}{models_filepath}l2_ridge_logistic.pkl', 'rb'))
l1_model = pickle.load(open(f'{project_path}{models_filepath}l1_lasso_logistic.pkl', 'rb'))
elasticnet_model = pickle.load(open(f'{project_path}{models_filepath}elasticnet_logistic.pkl', 'rb'))

print("Models loaded successfully!")

Models loaded successfully!


## 2. Understanding Hyperparameter Tuning

### 2.1 What is GridSearchCV?

**GridSearchCV** is a method for systematically working through multiple combinations of hyperparameter values, cross-validating as it goes, to determine which combination gives the best performance.

**How it works:**
1. Define a grid of hyperparameter values to try
2. For each combination, perform k-fold cross-validation
3. Compute the average score across folds
4. Select the combination with the best average score

**Why use GridSearchCV instead of manual tuning?**
- Exhaustively searches all combinations
- Uses cross-validation to avoid overfitting to validation data
- Automatically tracks results for comparison

### 2.2 Key Hyperparameters to Tune

| Hyperparameter | Description | Range to Search |
|:---------------|:------------|:----------------|
| `C` | Inverse of regularization strength | 0.001 to 100 (log scale) |
| `l1_ratio` | ElasticNet mixing (0=L2, 1=L1) | 0.0 to 1.0 |

**Important Notes:**
- **Small C** = Strong regularization (simpler model, may underfit)
- **Large C** = Weak regularization (complex model, may overfit)
- We search on a **logarithmic scale** because the effect of C is multiplicative

## 3. Tune the C Parameter

### 3.1 L2 (Ridge) Model Tuning

First, we tune the `C` parameter for the L2 regularized model.

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, make_scorer

# Define 5-fold StratifiedKFold for cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define F1 scorer for the minority class 'N'
f1_scorer = make_scorer(f1_score, pos_label='N')

# Define C values to search (logarithmic scale)
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

# Create parameter grid for L2 model
# Note: parameters inside pipeline use 'step__parameter' naming
l2_param_grid = {
    'classifier__C': C_values
}

print(f"L2 Parameter Grid:")
print(f"  C values: {C_values}")
print(f"  Total combinations to try: {len(C_values)}")

# Run GridSearchCV for L2 model
print("\nTuning L2 (Ridge) model...")

l2_grid_search = GridSearchCV(
    estimator=l2_model,
    param_grid=l2_param_grid,
    cv=cv,
    scoring=f1_scorer,
    return_train_score=True,
    n_jobs=-1,  # Use all available cores
    verbose=1
)

l2_grid_search.fit(X_train, y_train)

print(f"\nBest C value: {l2_grid_search.best_params_['classifier__C']}")
print(f"Best CV F1 score: {l2_grid_search.best_score_:.4f}")

# View all L2 results
l2_results = pd.DataFrame(l2_grid_search.cv_results_)
l2_results_display = l2_results[[
    'param_classifier__C', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score')

l2_results_display.columns = ['C', 'Train F1 (Mean)', 'CV F1 (Mean)', 'CV F1 (Std)', 'Rank']
print("\nL2 (Ridge) GridSearch Results:")
display(l2_results_display)

L2 Parameter Grid:
  C values: [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
  Total combinations to try: 6

Tuning L2 (Ridge) model...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Best C value: 100.0
Best CV F1 score: 0.4934

L2 (Ridge) GridSearch Results:


,C,Train F1 (Mean),CV F1 (Mean),CV F1 (Std),Rank
5,100.000,0.497164,0.493447,0.008826,1
3,1.000,0.496555,0.493373,0.009295,2
4,10.000,0.497217,0.493229,0.009156,3
2,0.100,0.495087,0.492487,0.009523,4
1,0.010,0.493322,0.489806,0.009796,5
0,0.001,0.478530,0.478214,0.009213,6


### 3.2 L1 (Lasso) Model Tuning

Now we tune the L1 regularized model.

In [7]:
# Create parameter grid for L1 model
l1_param_grid = {
    'classifier__C': C_values
}

print(f"L1 Parameter Grid:")
print(f"  C values: {C_values}")
print(f"  Total combinations to try: {len(C_values)}")

# Run GridSearchCV for L1 model
print("\nTuning L1 (Lasso) model...")

l1_grid_search = GridSearchCV(
    estimator=l1_model,
    param_grid=l1_param_grid,
    cv=cv,
    scoring=f1_scorer,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

l1_grid_search.fit(X_train, y_train)

print(f"\nBest C value: {l1_grid_search.best_params_['classifier__C']}")
print(f"Best CV F1 score: {l1_grid_search.best_score_:.4f}")

# View all L1 results
l1_results = pd.DataFrame(l1_grid_search.cv_results_)
l1_results_display = l1_results[[
    'param_classifier__C', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score')

l1_results_display.columns = ['C', 'Train F1 (Mean)', 'CV F1 (Mean)', 'CV F1 (Std)', 'Rank']
print("\nL1 (Lasso) GridSearch Results:")
display(l1_results_display)

L1 Parameter Grid:
  C values: [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
  Total combinations to try: 6

Tuning L1 (Lasso) model...
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Best C value: 0.01
Best CV F1 score: 0.5001

L1 (Lasso) GridSearch Results:


,C,Train F1 (Mean),CV F1 (Mean),CV F1 (Std),Rank
1,0.010,0.501108,0.500081,0.012000,1
2,0.100,0.495415,0.495274,0.009833,2
3,1.000,0.496989,0.494215,0.009148,3
5,100.000,0.497181,0.493514,0.008851,4
4,10.000,0.497147,0.493296,0.009182,5
0,0.001,0.465338,0.462642,0.007557,6


## 4. Tune ElasticNet Parameters

### 4.1 Tuning C and l1_ratio Together

ElasticNet has two hyperparameters to tune:
- **C**: Regularization strength
- **l1_ratio**: Balance between L1 and L2 penalties
  - `l1_ratio=0`: Pure L2 (Ridge)
  - `l1_ratio=1`: Pure L1 (Lasso)
  - `l1_ratio=0.5`: Equal mix

We use a 2D grid search to find the best combination.

In [8]:
# Define l1_ratio values to search
l1_ratio_values = [0.1, 0.3, 0.5, 0.7, 0.9]

# Create parameter grid for ElasticNet model
elasticnet_param_grid = {
    'classifier__C': C_values,
    'classifier__l1_ratio': l1_ratio_values
}

print(f"ElasticNet Parameter Grid:")
print(f"  C values: {C_values}")
print(f"  l1_ratio values: {l1_ratio_values}")
print(f"  Total combinations to try: {len(C_values) * len(l1_ratio_values)}")

# Run GridSearchCV for ElasticNet model
print("\nTuning ElasticNet model (this may take a moment)...")

elasticnet_grid_search = GridSearchCV(
    estimator=elasticnet_model,
    param_grid=elasticnet_param_grid,
    cv=cv,
    scoring=f1_scorer,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

elasticnet_grid_search.fit(X_train, y_train)

print(f"\nBest parameters:")
print(f"  C: {elasticnet_grid_search.best_params_['classifier__C']}")
print(f"  l1_ratio: {elasticnet_grid_search.best_params_['classifier__l1_ratio']}")
print(f"Best CV F1 score: {elasticnet_grid_search.best_score_:.4f}")

# View all ElasticNet results
elasticnet_results = pd.DataFrame(elasticnet_grid_search.cv_results_)
elasticnet_results_display = elasticnet_results[[
    'param_classifier__C', 'param_classifier__l1_ratio',
    'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score'
]].sort_values('rank_test_score').head(10)

elasticnet_results_display.columns = ['C', 'l1_ratio', 'Train F1 (Mean)', 'CV F1 (Mean)', 'CV F1 (Std)', 'Rank']
print("\nElasticNet GridSearch Results (Top 10):")
display(elasticnet_results_display)

ElasticNet Parameter Grid:
  C values: [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
  l1_ratio values: [0.1, 0.3, 0.5, 0.7, 0.9]
  Total combinations to try: 30

Tuning ElasticNet model (this may take a moment)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best parameters:
  C: 0.01
  l1_ratio: 0.9
Best CV F1 score: 0.4973

ElasticNet GridSearch Results (Top 10):


,C,l1_ratio,Train F1 (Mean),CV F1 (Mean),CV F1 (Std),Rank
9,0.01,0.9,0.498252,0.497300,0.012079,1
8,0.01,0.7,0.495425,0.495467,0.011680,2
14,0.10,0.9,0.495503,0.494985,0.010386,3
13,0.10,0.7,0.495192,0.494294,0.009433,4
19,1.00,0.9,0.496936,0.494012,0.009144,5
18,1.00,0.7,0.496832,0.493868,0.009048,6
17,1.00,0.5,0.496939,0.493798,0.009045,7
11,0.10,0.3,0.495358,0.493743,0.009856,8
16,1.00,0.3,0.496764,0.493652,0.009091,9
12,0.10,0.5,0.495230,0.493605,0.010081,10


### 4.2 Visualize Hyperparameter Grid

Let's visualize how different combinations of C and l1_ratio affect model performance.

## 5. Select the Best Model

### 5.1 Compare All Tuned Models

Now let's compare the best performing model from each regularization type.

### 5.2 Final Model Selection

We select the model with the highest cross-validation F1 score. In case of a tie, we prefer simpler models (fewer features or stronger regularization).

In [14]:
best_l2_c = l2_grid_search.best_params_['classifier__C']
best_l2_f1 = l2_grid_search.best_score_
# Get the std dev for the best C value
best_l2_std = l2_results[l2_results['param_classifier__C'] == best_l2_c]['std_test_score'].values[0]

best_l1_c = l1_grid_search.best_params_['classifier__C']
best_l1_f1 = l1_grid_search.best_score_
# Get the std dev for the best C value
best_l1_std = l1_results[l1_results['param_classifier__C'] == best_l1_c]['std_test_score'].values[0]

best_elasticnet_c = elasticnet_grid_search.best_params_['classifier__C']
best_elasticnet_l1_ratio = elasticnet_grid_search.best_params_['classifier__l1_ratio']
best_elasticnet_f1 = elasticnet_grid_search.best_score_
# Get the std dev for the best C and l1_ratio values
best_elasticnet_std = elasticnet_results[
    (elasticnet_results['param_classifier__C'] == best_elasticnet_c) &
    (elasticnet_results['param_classifier__l1_ratio'] == best_elasticnet_l1_ratio)
]['std_test_score'].values[0]

comparison_data = [
    {
        'Model': 'L2 (Ridge)',
        'Best C': best_l2_c,
        'Best l1_ratio': np.nan, # Not applicable for L2
        'CV F1 (Mean)': best_l2_f1,
        'CV F1 (Std)': best_l2_std
    },
    {
        'Model': 'L1 (Lasso)',
        'Best C': best_l1_c,
        'Best l1_ratio': np.nan, # Not applicable for L1
        'CV F1 (Mean)': best_l1_f1,
        'CV F1 (Std)': best_l1_std
    },
    {
        'Model': 'ElasticNet',
        'Best C': best_elasticnet_c,
        'Best l1_ratio': best_elasticnet_l1_ratio,
        'CV F1 (Mean)': best_elasticnet_f1,
        'CV F1 (Std)': best_elasticnet_std
    }
]

comparison_df = pd.DataFrame(comparison_data)

print("Comparison of Best Models:")
display(comparison_df)

# Create a bar chart comparing CV F1 scores with error bars
fig = px.bar(
    comparison_df,
    x='Model',
    y='CV F1 (Mean)',
    error_y='CV F1 (Std)',
    color='Model',
    title='Comparison of Best Model F1 Scores (with Error Bars)',
    labels={'CV F1 (Mean)': 'CV F1 Score'},
    color_discrete_sequence=px.colors.qualitative.Plotly # Use distinct colors
)

fig.update_layout(
    yaxis_title='Cross-Validation F1 Score',
    xaxis_title='Model Type',
    showlegend=False, # Legend is redundant with color mapping on x-axis
    height=500
)

fig.show()


Comparison of Best Models:


,Model,Best C,Best l1_ratio,CV F1 (Mean),CV F1 (Std)
0,L2 (Ridge),100.00,NaN,0.493447,0.008826
1,L1 (Lasso),0.01,NaN,0.500081,0.012000
2,ElasticNet,0.01,0.9,0.497300,0.012079


## 6. Evaluate on Test Set

Now that we've selected the best model using cross-validation, we evaluate it on the **held-out test set**. This gives us an unbiased estimate of how the model will perform on new data.

**Important**: We only evaluate on the test set ONCE after all tuning is complete to avoid data leakage.

### 6.1 Final Performance Metrics

In [15]:
best_model_name = comparison_df.loc[comparison_df['CV F1 (Mean)'].idxmax(), 'Model']

if best_model_name == 'L2 (Ridge)':
    best_model = l2_grid_search.best_estimator_
elif best_model_name == 'L1 (Lasso)':
    best_model = l1_grid_search.best_estimator_
elif best_model_name == 'ElasticNet':
    best_model = elasticnet_grid_search.best_estimator_
else:
    raise ValueError("Best model not found or unrecognized.")

print(f"Selected Best Model: {best_model_name}")

# Generate predictions on the test set
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1] # Probability of the positive class ('N')

# Calculate evaluation metrics for the test set
test_f1 = f1_score(y_test, y_pred, pos_label='N')
test_precision = precision_score(y_test, y_pred, pos_label='N')
test_recall = recall_score(y_test, y_pred, pos_label='N')
test_accuracy = accuracy_score(y_test, y_pred)

print("\n--- Test Set Performance --- ")
print(f"Model: {best_model_name}")
print(f"F1 Score (Minority Class 'N'): {test_f1:.4f}")
print(f"Precision (Minority Class 'N'): {test_precision:.4f}")
print(f"Recall (Minority Class 'N'): {test_recall:.4f}")
print(f"Overall Accuracy: {test_accuracy:.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

Selected Best Model: L1 (Lasso)

--- Test Set Performance --- 
Model: L1 (Lasso)
F1 Score (Minority Class 'N'): 0.5620
Precision (Minority Class 'N'): 0.4673
Recall (Minority Class 'N'): 0.7049
Overall Accuracy: 0.8311

--- Classification Report ---
              precision    recall  f1-score   support

           E       0.94      0.85      0.90      4516
           N       0.47      0.70      0.56       820

    accuracy                           0.83      5336
   macro avg       0.70      0.78      0.73      5336
weighted avg       0.87      0.83      0.84      5336



### 6.2 Confusion Matrix

In [20]:
from sklearn.metrics import confusion_matrix
import plotly.express as px

# Calculate the confusion matrix
# Labels are ['N', 'E'] where 'N' is the positive class (Left) and 'E' is the negative class (Stayed)
cm = confusion_matrix(y_test, y_pred, labels=['N', 'E'])

# Define labels for the heatmap axes
cm_display_labels = ['Left (N)', 'Stayed (E)']

# Create an interactive confusion matrix heatmap
fig = px.imshow(cm,
                text_auto=True,
                labels=dict(x="Predicted", y="Actual", color="Count"),
                x=cm_display_labels,
                y=cm_display_labels,
                color_continuous_scale="Blues",
                title="Confusion Matrix for Best Model (Test Set)",
                height=400)

fig.update_layout(
    xaxis_title="Predicted Class",
    yaxis_title="Actual Class"
)
fig.show()

# Interpret the confusion matrix based on labels=['N', 'E']
# cm[0, 0] = True Positives (Actual N, Predicted N) -> Actual Left, Predicted Left
# cm[0, 1] = False Negatives (Actual N, Predicted E) -> Actual Left, Predicted Stayed
# cm[1, 0] = False Positives (Actual E, Predicted N) -> Actual Stayed, Predicted Left
# cm[1, 1] = True Negatives (Actual E, Predicted E) -> Actual Stayed, Predicted Stayed

true_positives = cm[0, 0]
true_negatives = cm[1, 1]
false_positives = cm[1, 0]
false_negatives = cm[0, 1]

print("\n--- Confusion Matrix Interpretation ---")
print(f"True Positives (Actual Left 'N', Predicted Left 'N'): {true_positives}")
print(f"True Negatives (Actual Stayed 'E', Predicted Stayed 'E'): {true_negatives}")
print(f"False Positives (Actual Stayed 'E', Predicted Left 'N'): {false_positives}")
print(f"False Negatives (Actual Left 'N', Predicted Stayed 'E'): {false_negatives}")


--- Confusion Matrix Interpretation ---
True Positives (Actual Left 'N', Predicted Left 'N'): 578
True Negatives (Actual Stayed 'E', Predicted Stayed 'E'): 3857
False Positives (Actual Stayed 'E', Predicted Left 'N'): 659
False Negatives (Actual Left 'N', Predicted Stayed 'E'): 242


## 7. Visualize Hyperparameter Performance

Let's create comprehensive visualizations to understand how hyperparameters affect model performance.

In [23]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Prepare Data for Plotting ---

# L2 (Ridge) results
l2_results['log_C'] = np.log10(l2_results['param_classifier__C'])

# L1 (Lasso) results
l1_results['log_C'] = np.log10(l1_results['param_classifier__C'])

# ElasticNet: Use the 'elasticnet_results_best_l1' dataframe which already has
# the best l1_ratio for each C value.
elasticnet_results_best_l1['log_C'] = np.log10(elasticnet_results_best_l1['param_classifier__C'])

# --- Create Subplots ---
fig = make_subplots(rows=3, cols=1,
                    subplot_titles=("L2 (Ridge) Regularization",
                                    "L1 (Lasso) Regularization",
                                    "ElasticNet Regularization"),
                    vertical_spacing=0.15)

# --- Add L2 (Ridge) Trace ---
fig.add_trace(
    go.Scatter(
        x=l2_results['log_C'],
        y=l2_results['mean_test_score'],
        mode='lines+markers',
        name='L2 CV F1 (Mean)',
        error_y=dict(
            type='data',
            array=l2_results['std_test_score'],
            visible=True
        ),
        hovertemplate='<b>C (log10):</b> %{x:.2f}<br><b>C:</b> %{customdata[0]:.3f}<br><b>Mean CV F1:</b> %{y:.4f}<br><b>Std CV F1:</b> %{customdata[1]:.4f}<extra></extra>',
        customdata=l2_results[['param_classifier__C', 'std_test_score']].values
    ),
    row=1, col=1
)

# --- Add L1 (Lasso) Trace ---
fig.add_trace(
    go.Scatter(
        x=l1_results['log_C'],
        y=l1_results['mean_test_score'],
        mode='lines+markers',
        name='L1 CV F1 (Mean)',
        error_y=dict(
            type='data',
            array=l1_results['std_test_score'],
            visible=True
        ),
        hovertemplate='<b>C (log10):</b> %{x:.2f}<br><b>C:</b> %{customdata[0]:.3f}<br><b>Mean CV F1:</b> %{y:.4f}<br><b>Std CV F1:</b> %{customdata[1]:.4f}<extra></extra>',
        customdata=l1_results[['param_classifier__C', 'std_test_score']].values
    ),
    row=2, col=1
)

# --- Add ElasticNet Trace ---
fig.add_trace(
    go.Scatter(
        x=elasticnet_results_best_l1['log_C'],
        y=elasticnet_results_best_l1['mean_test_score'],
        mode='lines+markers',
        name='ElasticNet CV F1 (Mean)',
        error_y=dict(
            type='data',
            array=elasticnet_results_best_l1['std_test_score'],
            visible=True
        ),
        hovertemplate='<b>C (log10):</b> %{x:.2f}<br><b>C:</b> %{customdata[0]:.3f}<br><b>Best l1_ratio:</b> %{customdata[1]:.1f}<br><b>Mean CV F1:</b> %{y:.4f}<br><b>Std CV F1:</b> %{customdata[2]:.4f}<extra></extra>',
        customdata=elasticnet_results_best_l1[['param_classifier__C', 'param_classifier__l1_ratio', 'std_test_score']].values
    ),
    row=3, col=1
)

# --- Update Layout and Axes ---
fig.update_layout(
    title_text="Cross-Validation F1 Score vs. Regularization Strength (C)",
    height=900,
    showlegend=False
)

# Update x-axes to show C values more interpretably (e.g., 10^-3, 10^-2, etc.)
for i in range(1, 4):
    fig.update_xaxes(
        title_text="C (log10)",
        row=i, col=1,
        tickvals=np.log10(C_values),
        ticktext=[f'10^{{{val:.0f}}}' if val != 0 else '1' for val in np.log10(C_values)],
        type='linear'
    )
    fig.update_yaxes(title_text="CV F1 Score", row=i, col=1, range=[0.4, 0.55])

fig.show()

**Interpretation:**

- When training and validation curves are close together, the model generalizes well
- A large gap (training >> validation) indicates overfitting
- Strong regularization (small C) prevents overfitting but may underfit
- The optimal C balances bias and variance

## 8. Summary

In this notebook, we tuned regularization hyperparameters using GridSearchCV and evaluated the best model on the test set.

### Key Findings

In [24]:
import pickle

# 1. Save the best tuned model
best_model_filename = f'{project_path}{models_filepath}best_tuned_logistic_model.pkl'
with open(best_model_filename, 'wb') as file:
    pickle.dump(best_model, file)
print(f"Best tuned model saved to: {best_model_filename}")

# 2. Save a dictionary containing all GridSearchCV objects and the best model name
all_grid_searches = {
    'l2_grid_search': l2_grid_search,
    'l1_grid_search': l1_grid_search,
    'elasticnet_grid_search': elasticnet_grid_search,
    'best_model_name': best_model_name
}

comparison_data_filename = f'{project_path}{models_filepath}grid_search_comparison_data.pkl'
with open(comparison_data_filename, 'wb') as file:
    pickle.dump(all_grid_searches, file)
print(f"GridSearchCV objects and best model name saved to: {comparison_data_filename}")

Best tuned model saved to: /content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/models/best_tuned_logistic_model.pkl
GridSearchCV objects and best model name saved to: /content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3/models/grid_search_comparison_data.pkl


### Key Takeaways

1. **GridSearchCV** systematically searches hyperparameter combinations using cross-validation

2. **The C parameter** controls regularization strength:
   - Small C = strong regularization (simpler model)
   - Large C = weak regularization (more complex model)

3. **ElasticNet's l1_ratio** controls the balance between L1 and L2 penalties

4. **Visualizations** help understand how hyperparameters affect performance

5. **Test set evaluation** should only be done once, after all tuning is complete

### What We Learned About Our Data

| Regularization Type | Best Performance | What This Suggests |
|:--------------------|:-----------------|:------------------|
| L2 (Ridge) | Good | All features contribute somewhat |
| L1 (Lasso) | Good | Some feature selection beneficial |
| ElasticNet | Best of both | Combination of shrinkage and selection helps |

### Practical Implications

- The tuned model can be used to identify at-risk students early
- Regularization helped improve generalization beyond the default settings
- The model's interpretability is maintained while improving performance

### Next Steps

Congratulations on completing Module 2 of Course 3! You have learned how to:

1. Understand regularization and its importance (Notebook 2.1)
2. Build regularized logistic regression models (Notebook 2.2)
3. Train and compare different regularization types (Notebook 2.3)
4. Tune hyperparameters and select the best model (This notebook)

In the next module, we will explore **tree-based models** — Decision Trees, Random Forests, and XGBoost — powerful algorithms that can capture non-linear relationships and feature interactions automatically.

**Proceed to:** `3.1 Introduction to Tree-Based Models`